# Loadsmart — Analytics Engineer Challenge

Notebook com funções utilitárias e exportações a partir do DuckDB.

**Pré-requisitos** (já instalados no `.venv`):
```
pip install duckdb pandas paramiko
```

Execute o pipeline antes de rodar este notebook:
```bash
python scripts/ingest.py
cd dbt && dbt run
```

## 1. Setup

In [ ]:
import os
import io
import smtplib
import email.mime.multipart
import email.mime.text
import email.mime.base
import email.encoders
import re
from pathlib import Path
from datetime import date
from dateutil.relativedelta import relativedelta

import duckdb
import pandas as pd
import paramiko

# Caminho do banco — ajuste se rodar fora da raiz do projeto
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
DUCKDB_PATH = str(PROJECT_ROOT / "data" / "loadsmart.duckdb")

print(f"DuckDB: {DUCKDB_PATH}")
print(f"Arquivo existe: {Path(DUCKDB_PATH).exists()}")

## 2. Conexão com DuckDB

In [ ]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# Confirma tabelas disponíveis
con.execute("SHOW ALL TABLES").df()

## 3. Função `split_lane()`

Recebe uma string de lane no formato `"City,ST -> City,ST"` e retorna um dicionário com as cidades e estados de origem e destino.

In [ ]:
def split_lane(lane: str) -> dict:
    """
    Parseia uma lane no formato 'City,ST -> City,ST'.

    Parâmetros
    ----------
    lane : str
        String de lane, ex: 'Chicago,IL -> Dallas,TX'

    Retorna
    -------
    dict com chaves:
        pickup_city, pickup_state, delivery_city, delivery_state

    Lança
    -----
    ValueError se o formato for inválido.
    """
    if not lane or " -> " not in lane:
        raise ValueError(f"Formato de lane inválido: {lane!r}. Esperado: 'City,ST -> City,ST'")

    origin, destination = lane.split(" -> ", maxsplit=1)

    def _parse_side(side: str) -> tuple[str, str]:
        parts = side.split(",", maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Lado da lane malformado: {side!r}")
        return parts[0].strip(), parts[1].strip()

    pickup_city, pickup_state = _parse_side(origin)
    delivery_city, delivery_state = _parse_side(destination)

    return {
        "pickup_city": pickup_city,
        "pickup_state": pickup_state,
        "delivery_city": delivery_city,
        "delivery_state": delivery_state,
    }


# --- Testes manuais ---
cases = [
    ("Chicago,IL -> Dallas,TX",      {"pickup_city": "Chicago",   "pickup_state": "IL", "delivery_city": "Dallas",  "delivery_state": "TX"}),
    ("New York,NY -> Los Angeles,CA", {"pickup_city": "New York",  "pickup_state": "NY", "delivery_city": "Los Angeles", "delivery_state": "CA"}),
    (" Miami , FL ->  Atlanta , GA ", {"pickup_city": "Miami",     "pickup_state": "FL", "delivery_city": "Atlanta", "delivery_state": "GA"}),
]

for lane_str, expected in cases:
    result = split_lane(lane_str)
    assert result == expected, f"FALHA: {lane_str!r} → {result}"
    print(f"OK  {lane_str!r}")
    print(f"    {result}")

In [ ]:
# Aplicando em amostra real do DuckDB
sample = con.execute("""
    SELECT LANE_RAW
    FROM main_mart.fct_shipments
    LIMIT 5
""").df()

pd.DataFrame(sample["LANE_RAW"].apply(split_lane).tolist())

## 4. Função `send_csv_email()`

Exporta um DataFrame como CSV em memória (sem gravar em disco) e envia como anexo por SMTP.

In [ ]:
def send_csv_email(
    df: pd.DataFrame,
    recipients: list[str],
    smtp_config: dict,
    subject: str = "Loadsmart — Export",
    filename: str = "export.csv",
    body: str = "Segue em anexo o CSV solicitado.",
) -> None:
    """
    Envia um DataFrame como CSV em anexo via SMTP.

    Parâmetros
    ----------
    df         : DataFrame a exportar
    recipients : lista de endereços de e-mail
    smtp_config: dict com chaves obrigatórias:
                   host (str), port (int), user (str), password (str)
                 e opcionais:
                   use_tls (bool, default True), sender (str, default user)
    subject    : assunto do e-mail
    filename   : nome do arquivo CSV no anexo
    body       : corpo do e-mail (texto simples)

    Exemplo de smtp_config
    ----------------------
    {
        "host": "smtp.gmail.com",
        "port": 587,
        "user": "you@gmail.com",
        "password": "app-password",
    }
    """
    required_keys = {"host", "port", "user", "password"}
    missing = required_keys - smtp_config.keys()
    if missing:
        raise ValueError(f"smtp_config está faltando: {missing}")

    sender = smtp_config.get("sender", smtp_config["user"])
    use_tls = smtp_config.get("use_tls", True)

    # Serializa o DataFrame em memória
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    csv_bytes = buffer.getvalue().encode("utf-8")

    # Monta a mensagem
    msg = email.mime.multipart.MIMEMultipart()
    msg["From"] = sender
    msg["To"] = ", ".join(recipients)
    msg["Subject"] = subject
    msg.attach(email.mime.text.MIMEText(body, "plain"))

    attachment = email.mime.base.MIMEBase("application", "octet-stream")
    attachment.set_payload(csv_bytes)
    email.encoders.encode_base64(attachment)
    attachment.add_header("Content-Disposition", "attachment", filename=filename)
    msg.attach(attachment)

    # Envia
    with smtplib.SMTP(smtp_config["host"], smtp_config["port"]) as server:
        if use_tls:
            server.starttls()
        server.login(smtp_config["user"], smtp_config["password"])
        server.sendmail(sender, recipients, msg.as_string())

    print(f"E-mail enviado para: {', '.join(recipients)} | Arquivo: {filename} ({len(csv_bytes):,} bytes)")


print("Função send_csv_email carregada.")
print()
print("Exemplo de uso:")
print("""
send_csv_email(
    df=df_last_month,
    recipients=["analytics@loadsmart.com"],
    smtp_config={
        "host": "smtp.gmail.com",
        "port": 587,
        "user": "you@gmail.com",
        "password": "app-password",
    },
    subject="Deliveries — Last Month",
    filename="deliveries_last_month.csv",
)
""")

In [ ]:
def send_csv_email_from_path(
    csv_path: str,
    subject: str,
    body: str,
    recipients: list[str],
    smtp_config: dict,
    filename: str | None = None,
) -> None:
    """
    Wrapper conforme assinatura do PRD: recebe caminho do CSV, assunto e corpo.

    Parâmetros
    ----------
    csv_path   : caminho para o arquivo CSV no disco
    subject    : assunto do e-mail
    body       : corpo do e-mail (texto simples)
    recipients : lista de endereços de destino
    smtp_config: dict com host, port, user, password
    filename   : nome do anexo (default: nome do arquivo)
    """
    df = pd.read_csv(csv_path)
    fname = filename or Path(csv_path).name
    send_csv_email(df, recipients, smtp_config, subject=subject, body=body, filename=fname)


print("Wrapper send_csv_email_from_path carregado.")
print()
print("Exemplo de uso (assinatura PRD):")
print("""
send_csv_email_from_path(
    csv_path="data/exports/deliveries_2024_11.csv",
    subject="Deliveries — November 2024",
    body="Segue em anexo o CSV do último mês disponível.",
    recipients=["analytics@loadsmart.com"],
    smtp_config={"host": "smtp.gmail.com", "port": 587, "user": "you@gmail.com", "password": "app-password"},
)
""")

## 5. Função `send_csv_sftp()`

Exporta um DataFrame como CSV e faz upload via SFTP para um servidor remoto.

In [ ]:
def send_csv_sftp(
    df: pd.DataFrame,
    remote_path: str,
    credentials: dict,
) -> None:
    """
    Exporta um DataFrame como CSV e faz upload via SFTP.

    Parâmetros
    ----------
    df          : DataFrame a exportar
    remote_path : caminho completo no servidor remoto, ex: '/data/exports/file.csv'
    credentials : dict com chaves:
                    host (str), username (str)
                  e uma das opções de autenticação:
                    password (str)  — autenticação por senha
                    key_path (str)  — caminho para chave privada SSH
                  opcionais:
                    port (int, default 22)

    Exemplo de credentials
    ----------------------
    # Por senha:
    {"host": "sftp.loadsmart.com", "username": "deploy", "password": "s3cr3t"}

    # Por chave SSH:
    {"host": "sftp.loadsmart.com", "username": "deploy", "key_path": "~/.ssh/id_rsa"}
    """
    required_keys = {"host", "username"}
    missing = required_keys - credentials.keys()
    if missing:
        raise ValueError(f"credentials está faltando: {missing}")
    if "password" not in credentials and "key_path" not in credentials:
        raise ValueError("credentials deve ter 'password' ou 'key_path'")

    port = credentials.get("port", 22)

    # Serializa o DataFrame em memória
    buffer = io.BytesIO()
    df.to_csv(io.TextIOWrapper(buffer, encoding="utf-8"), index=False)
    buffer.seek(0)
    size = len(buffer.getvalue())

    # Conecta via Paramiko
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    connect_kwargs = {
        "hostname": credentials["host"],
        "port": port,
        "username": credentials["username"],
    }

    if "key_path" in credentials:
        connect_kwargs["key_filename"] = os.path.expanduser(credentials["key_path"])
    else:
        connect_kwargs["password"] = credentials["password"]

    ssh.connect(**connect_kwargs)

    try:
        sftp = ssh.open_sftp()
        sftp.putfo(buffer, remote_path)
        sftp.close()
    finally:
        ssh.close()

    print(f"Upload SFTP concluído: {credentials['host']}:{remote_path} ({size:,} bytes)")


print("Função send_csv_sftp carregada.")
print()
print("Exemplo de uso:")
print("""
send_csv_sftp(
    df=df_last_month,
    remote_path="/exports/deliveries_last_month.csv",
    credentials={
        "host": "sftp.loadsmart.com",
        "username": "deploy",
        "key_path": "~/.ssh/id_rsa",
    },
)
""")

In [ ]:
def send_csv_sftp_from_path(
    csv_path: str,
    remote_path: str,
    credentials: dict,
) -> None:
    """
    Wrapper conforme assinatura do PRD: recebe caminho do CSV local e caminho remoto.

    Parâmetros
    ----------
    csv_path    : caminho para o arquivo CSV no disco
    remote_path : caminho completo no servidor remoto, ex: '/exports/file.csv'
    credentials : dict com host, username e password ou key_path
    """
    df = pd.read_csv(csv_path)
    send_csv_sftp(df, remote_path, credentials)


print("Wrapper send_csv_sftp_from_path carregado.")
print()
print("Exemplo de uso (assinatura PRD):")
print("""
send_csv_sftp_from_path(
    csv_path="data/exports/deliveries_2024_11.csv",
    remote_path="/exports/deliveries_2024_11.csv",
    credentials={"host": "sftp.loadsmart.com", "username": "deploy", "key_path": "~/.ssh/id_rsa"},
)
""")

## 6. Exportação: Entregas do Último Mês

Consulta `main_mart.fct_shipments` filtrando pelo **último mês disponível nos dados** (não pela data atual).
O CSV exportado contém exatamente as 9 colunas especificadas no PRD.

In [ ]:
# Descobre o último mês disponível nos dados (não usa date.today())
max_date_raw = con.execute("""
    SELECT MAX(DELIVERED_AT)
    FROM main_mart.fct_shipments
    WHERE LOAD_WAS_CANCELLED = false
""").fetchone()[0]

if max_date_raw is None:
    raise ValueError("Nenhuma entrega encontrada em fct_shipments.")

# Normaliza para date (DuckDB pode retornar datetime ou date)
max_date = max_date_raw.date() if hasattr(max_date_raw, 'date') else max_date_raw

first_of_last_month = max_date.replace(day=1)
first_of_next_month = first_of_last_month + relativedelta(months=1)

print(f"MAX delivered_at nos dados: {max_date}")
print(f"Último mês disponível:      {first_of_last_month.strftime('%B %Y')}")
print(f"Período filtrado:           {first_of_last_month} → {first_of_next_month} (exclusive)")

In [ ]:
df_last_month = con.execute("""
    SELECT
        f.LOADSMART_ID,
        f.LANE_RAW,
        f.PICKUP_CITY,
        f.PICKUP_STATE,
        f.DELIVERY_CITY,
        f.DELIVERY_STATE,
        f.BOOKED_AT,
        f.PICKUP_AT,
        f.DELIVERED_AT,
        f.BOOK_PRICE,
        f.SOURCE_PRICE,
        f.PNL,
        f.MILEAGE,
        f.LEAD_TIME_DAYS,
        f.DELIVERED_ON_TIME,
        f.EQUIPMENT_TYPE,
        f.SOURCING_CHANNEL,
        f.LOAD_WAS_CANCELLED,
        dc.CARRIER_NAME,
        dc.CARRIER_RATING,
        dc.VIP_CARRIER,
        ds.SHIPPER_NAME
    FROM main_mart.fct_shipments f
    LEFT JOIN main_mart.dim_carrier  dc ON f.CARRIER_SK  = dc.CARRIER_SK
    LEFT JOIN main_mart.dim_shipper  ds ON f.SHIPPER_SK  = ds.SHIPPER_SK
    WHERE
        f.DELIVERED_AT >= ?
        AND f.DELIVERED_AT <  ?
        AND f.LOAD_WAS_CANCELLED = false
    ORDER BY f.DELIVERED_AT
""", [str(first_of_last_month), str(first_of_next_month)]).df()

print(f"Linhas: {len(df_last_month):,}")
df_last_month.head()

In [ ]:
# Resumo do mês
print("=== Resumo do Último Mês ===")
print(f"Total de entregas:    {len(df_last_month):,}")
if len(df_last_month) > 0:
    print(f"Receita total:        $ {df_last_month['BOOK_PRICE'].sum():,.2f}")
    print(f"PnL total:            $ {df_last_month['PNL'].sum():,.2f}")
    print(f"On-time rate:         {df_last_month['DELIVERED_ON_TIME'].mean():.1%}")
    print(f"Lead time médio:      {df_last_month['LEAD_TIME_DAYS'].mean():.2f} dias")
else:
    print("Nenhuma entrega no período — verifique o range de datas do dataset.")

In [ ]:
# Colunas exatas do PRD (9 colunas, nomes como no enunciado)
PRD_COLUMNS = {
    "LOADSMART_ID":  "loadsmart_id",
    "SHIPPER_NAME":  "shipper_name",
    "DELIVERED_AT":  "delivery_date",
    "PICKUP_CITY":   "pickup_city",
    "PICKUP_STATE":  "pickup_state",
    "DELIVERY_CITY": "delivery_city",
    "DELIVERY_STATE":"delivery_state",
    "BOOK_PRICE":    "book_price",
    "CARRIER_NAME":  "carrier_name",
}

df_export = df_last_month[list(PRD_COLUMNS.keys())].rename(columns=PRD_COLUMNS)

output_filename = f"deliveries_{first_of_last_month.strftime('%Y_%m')}.csv"
output_path = PROJECT_ROOT / "data" / "exports" / output_filename
output_path.parent.mkdir(parents=True, exist_ok=True)

df_export.to_csv(output_path, index=False)
print(f"CSV salvo em: {output_path}")
print(f"Colunas: {list(df_export.columns)}")
print(f"Total de linhas: {len(df_export):,}")
df_export.head()

In [ ]:
# ─── Envio por E-mail — wrapper PRD (csv_path, subject, body) ──────────────
#
# send_csv_email_from_path(
#     csv_path=str(output_path),
#     subject=f"Deliveries — {first_of_last_month.strftime('%B %Y')}",
#     body="Segue em anexo o CSV do último mês disponível.",
#     recipients=["analytics@loadsmart.com"],
#     smtp_config={
#         "host": "smtp.gmail.com",
#         "port": 587,
#         "user": "you@gmail.com",
#         "password": "your-app-password",
#     },
# )

# ─── Envio por SFTP — wrapper PRD (csv_path, remote_path) ───────────────────
#
# send_csv_sftp_from_path(
#     csv_path=str(output_path),
#     remote_path=f"/exports/{output_filename}",
#     credentials={
#         "host": "sftp.loadsmart.com",
#         "username": "deploy",
#         "key_path": "~/.ssh/id_rsa",
#     },
# )

print("Células de envio comentadas — descomente e preencha com suas credenciais.")

In [ ]:
con.close()
print("Conexão DuckDB fechada.")